In [1]:
import pandas as pd
import numpy as np

# --- 1. LOAD DATA ---
# Ensure 'AirQuality_Daily_StudentVersion(in).csv' is in the same folder
df = pd.read_csv('AirQuality_Daily_StudentVersion(in).csv')
df['date'] = pd.to_datetime(df['date'])

# --- 2. TOP 5 LOCATIONS (VOC, PM 2.5, PM 10.0) ---
# Grouping by sensor name to find mean and median as requested by the client
pollutants = ['voc', 'pm2.5_atm', 'pm10.0_atm']
for pollutant in pollutants:
    print(f"\n--- Top 5 Locations for {pollutant.upper()} (Mean) ---")
    print(df.groupby('sensor.name')[pollutant].mean().sort_values(ascending=False).head(5))
    
    print(f"\n--- Top 5 Locations for {pollutant.upper()} (Median) ---")
    print(df.groupby('sensor.name')[pollutant].median().sort_values(ascending=False).head(5))

# --- 3. MAXIMUM VALUES (WHEN AND WHERE) ---
# Identifying the specific days and locations where the highest pollution occurred
for pollutant in pollutants:
    max_row = df.loc[df[pollutant].idxmax()]
    print(f"\nMaximum {pollutant.upper()} of {max_row[pollutant]:.2f} occurred on {max_row['date'].date()} at {max_row['sensor.name']}")

# --- 4. HUMIDITY & TEMPERATURE ANALYSIS ---
# Categorizing humidity based on client criteria:
# Low (<50%), High (50-80%), Very High (>80%)
def categorize_humidity(h):
    if h < 50: return 'Low (<50%)'
    elif 50 <= h <= 80: return 'High (50-80%)'
    else: return 'Very High (>80%)'

df['humidity_category'] = df['humidity'].apply(categorize_humidity)

print("\n--- Impact of Humidity on PM 2.5 ---")
print(df.groupby('humidity_category')['pm2.5_atm'].mean())

# Correlation for Temperature and Altitude (Bonus)
print("\n--- Environmental Correlations (PM 2.5 vs Others) ---")
correlations = df[['pm2.5_atm', 'temperature', 'sensor.altitude']].corr()
print(correlations['pm2.5_atm'])

# --- 5. AQI HEALTH RISKS (UNHEALTHY FOR SENSITIVE GROUPS) ---
# Thresholds based on EPA: PM2.5 > 35.4 and PM10 > 154
aqi_pm25_risk = df[df['pm2.5_atm'] > 35.4]
aqi_pm10_risk = df[df['pm10.0_atm'] > 154]

print(f"\nTotal Health Risk Instances (PM 2.5): {len(aqi_pm25_risk)}")
if len(aqi_pm25_risk) > 0:
    print("Recent PM 2.5 Risk Events:")
    print(aqi_pm25_risk[['date', 'sensor.name', 'pm2.5_atm']].head())

print(f"\nTotal Health Risk Instances (PM 10.0): {len(aqi_pm10_risk)}")

# --- 6. EXPORTING RESULTS ---
# Saving the summary by location for your written report tables
location_summary = df.groupby('sensor.name')[pollutants].mean()
location_summary.to_csv('AirQuality_Location_Final_Results.csv')
print("\nSuccess: Analysis complete. Results saved to 'AirQuality_Location_Final_Results.csv'.")


--- Top 5 Locations for VOC (Mean) ---
sensor.name
Swnphd-ogallala                          399.434240
FCHD-YPS                                 372.462720
Three Rivers Public Health Department    370.216208
ELVPHD Norfolk HD 4                      360.833744
Swnphd-mccook                            353.941581
Name: voc, dtype: float64

--- Top 5 Locations for VOC (Median) ---
sensor.name
Swnphd-ogallala                          423.082292
Swnphd-mccook                            381.468479
Three Rivers Public Health Department    376.810167
FCHD-YPS                                 375.383500
ELVPHD Norfolk HD 4                      368.580500
Name: voc, dtype: float64

--- Top 5 Locations for PM2.5_ATM (Mean) ---
sensor.name
Broken Bow                                              928.710593
#16 - Richardson County Courthouse                      700.127342
#18 - Southeast District Health Department- Tecumseh    613.175352
NCDHD O'Neill #11                                       164.495